# TSL-51 Training

Train Thai Sign Language recognition models on Google Colab.

## 1. Setup (Run 01_setup.ipynb first)

In [ ]:
import os
import sys
import torch
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/TSL'
os.chdir(WORK_DIR)
sys.path.insert(0, os.path.join(WORK_DIR, 'src'))

print(f'Working directory: {os.getcwd()}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 2. Training Configuration

In [ ]:
# Training parameters
CONFIG = {
    # Dataset
    'dataset': 'tsl51_user_sign',    # Options: tsl51_user_sign, tsl51_expert, tsl51_expert_full, tsl51_combined
    'seq_mode': True,                 # True = use per-frame sequences (T=30), False = mean-aggregated
    'target_frames': 30,              # Fixed sequence length for seq_mode

    # Model
    'model_type': 'gru',             # Options: gru, mlp
    'hidden_dim': 256,
    'num_layers': 3,
    'dropout': 0.3,

    # Training
    'learning_rate': 0.001,
    'batch_size': 64,
    'epochs': 100,
    'k_folds': 5,
    'patience': 15,
    'test_split': 0.2,

    # Augmentation (seq_mode-aware: noise, scale, flip, noise_scale)
    'augment_factor': 5,
    'noise_level': 0.01,
    'scale_range': (0.95, 1.05),
}

print('Training configuration:')
for key, value in CONFIG.items():
    print(f'  {key}: {value}')

## 3. Prepare Canonical Pipeline Configuration

Map the notebook `CONFIG` dictionary onto the canonical `src.train.pipeline` entrypoint.

In [ ]:
from pathlib import Path

from src.train.pipeline import run_training_pipeline

PIPELINE_CONFIG = {
    'dataset': CONFIG['dataset'],
    'model': CONFIG['model_type'],
    'hidden_dim': CONFIG['hidden_dim'],
    'num_layers': CONFIG['num_layers'],
    'dropout': CONFIG['dropout'],
    'learning_rate': CONFIG['learning_rate'],
    'batch_size': CONFIG['batch_size'],
    'epochs': CONFIG['epochs'],
    'n_folds': CONFIG['k_folds'],
    'patience': CONFIG['patience'],
    'seed': CONFIG.get('seed', 42),
    'feature_level': CONFIG.get('feature_level', 'basic'),
    'seq_mode': CONFIG.get('seq_mode', True),
    'target_frames': CONFIG.get('target_frames', 30),
    'augmentation_factor': CONFIG.get('augment_factor', 0),
    'noise_level': CONFIG.get('noise_level', 0.01),
    'scale_range': tuple(CONFIG.get('scale_range', (0.95, 1.05))),
    'split_strategy': CONFIG.get('split_strategy', 'video_family_holdout'),
    'primary_metric': CONFIG.get('primary_metric', 'macro_f1'),
    'real_world_mode': CONFIG.get('real_world_mode', True),
    'include_augmented': CONFIG.get('include_augmented', False),
    'force_download': CONFIG.get('force_download', False),
    'no_cache': CONFIG.get('no_cache', False),
    'samples': CONFIG.get('samples'),
    'data_path': CONFIG.get('data_path'),
    'val_size': CONFIG.get('val_size', 0.15),
    'test_size': CONFIG.get('test_split', 0.2),
    'output_dir': Path(CONFIG.get('output_dir', 'artifacts/runs/colab_train')),
}

print('Canonical pipeline configuration:')
for key, value in PIPELINE_CONFIG.items():
    print(f'  {key}: {value}')

## 4. Run Canonical Training Pipeline

This calls `src.train.pipeline.run_training_pipeline(...)` so grouped splitting, train-only augmentation, normalization, training, and artifact persistence all come from the shared canonical path.

In [ ]:
result = run_training_pipeline(PIPELINE_CONFIG)
artifacts = result['artifacts']

print(f"Primary metric ({result['primary_metric_name']}): {result['primary_metric']:.2f}")
print(f"Checkpoint: {artifacts['checkpoint']}")
print(f"Metrics: {artifacts['metrics']}")
print(f"Split manifest: {artifacts['split_manifest']}")
print(f"Preprocessing manifest: {artifacts['preprocessing_manifest']}")

## 5. Training Complete

Proceed to evaluation notebook (`03_evaluate.ipynb`) with the checkpoint and canonical artifacts written by `src.train.pipeline`.